# For colab environment

In [1]:
# !pip install nltk transformers==4.35.0 torch==2.6.0 torchvision==0.21.0 datasets accelerate==0.24.0 huggingface==0.0.1 datasets==2.14.7

In [1]:
import torch 
print(torch.cuda.is_available())
print(torch.__version__)

True
2.6.0+cu124


In [3]:
# !git clone https://github.com/BernardMoy/NLP-PCL-Classification.git

In [4]:
# %cd NLP-PCL-Classification/

In [2]:
!nvidia-smi

Sun Mar  1 22:02:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA TITAN Xp                Off |   00000000:02:00.0  On |                  N/A |
| 23%   43C    P0             66W /  250W |     298MiB /  12288MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Load the validation and test dataset

In [ ]:
import pandas as pd 
import numpy as np 
from matplotlib import pyplot as plt
from collections import Counter

df = pd.read_csv('data/dontpatronizeme_pcl.tsv', sep='\t')

# Remove rows with NA labels 
df = df.dropna() 

# Add a bool_labels column for binary classification
df["bool_labels"] = df["label"] > 1   # is PCL if >1
val_labels = pd.read_csv('data/dev_semeval_parids-labels.csv')["par_id"]
df_val = df[df["par_id"].isin(val_labels)].reset_index() 

# read the test dataset 
df_test = pd.read_csv('data/task4_test.tsv', sep='\t')


# Text Cleaning

In [ ]:
# Remove special characters
SPECIAL_CHARACTERS = ['&amp;', '&lt;', '&gt;', '<h>', '\n', '\t']
for char in SPECIAL_CHARACTERS:
    df_val["text"] = df_val["text"].str.replace(char, "")
    df_test["text"] = df_test["text"].str.replace(char, "")

# Replace numbers with 0
df_val["text"] = df_val["text"].str.replace(r"\d+", "0", regex=True)
df_test["text"] = df_test["text"].str.replace(r"\d+", "0", regex=True)


People who are homeless , those who were once homeless , those working with the homeless and concerned New Zealanders are being asked to share their experiences and solutions to this growing issue with the Cross-Party Homelessness Inquiry . More
Council customers only signs would be displayed . Two of the spaces would be reserved for disabled persons and there would be five P0 spaces and eight P0 ones .


# Oversample the minority class
For each keyword category, inflate the number of positive examples to a certain percentage

In [5]:
POSITIVE_PERCENTAGE = 25


# Find all the unique keywords in the training dataset
keywords = pd.unique(df_train["keyword"])


# Extract the sub-dataset for each keyword
for keyword in keywords:
    subdata = df_train[df_train["keyword"] == keyword]
    rows = subdata.shape[0]


    # Find the number of positive entires x
    subdata_positive = subdata[subdata["bool_labels"] == True]
    positive_rows = subdata_positive.shape[0]


    # Calculate the number of additional samples needed to make the positive class reach the desired percentage
    # (p+x)/(r+x) = POS PERCENTAGE
    n_samples = round((100*positive_rows-POSITIVE_PERCENTAGE*rows)/(POSITIVE_PERCENTAGE-100)*1.0)


    # Sample with replacement from the sub dataset and add new rows
    sampled = subdata_positive.sample(n_samples, replace=True).reset_index(drop=True)
   
    # concat with the main df
    df_train = pd.concat([df_train, sampled], ignore_index=True)


df_train["bool_labels"].value_counts()


bool_labels
False    7581
True     2527
Name: count, dtype: int64

# Add contextual information to the text tokens

In [6]:
def add_info(df): 
    # Append the keyword and country code to the text, and separate them with additional separator tokens
    # Remove dashes in the keyword to match the format in the texts 
    return df["keyword"].str.replace('-', " ") + "<SEP>" + df["country_code"] + "<SEP>" + df["text_cr"]

# Tokenization

In [7]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification, AutoConfig, Trainer, TrainingArguments

tokenizer = RobertaTokenizer.from_pretrained("roberta-base") 

# define special tokens for separating text 
special_tokens = {"additional_special_tokens": ["<SEP>"]}
tokenizer.add_special_tokens(special_tokens) 

# Create text with contextual information 
def tokenize(df): 
    text_with_context = df["text"]

    encoding = tokenizer(
        text_with_context.tolist(), 
        padding="max_length",   # Add padding to shorter sentences 
        max_length=256,
        truncation = True, 
        return_attention_mask = True 
    )

    return encoding

/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


# Convert to pyTorch dataset

In [8]:
import torch 
from torch.utils.data import DataLoader, TensorDataset
from datasets import Dataset

def to_dataset(df): 
    # Obtain tokens (input_ids, attention_mask) from the dataset 
    encoding = tokenize(df) 

    # Return huggingface dataset 
    return Dataset.from_dict({
        "input_ids": encoding["input_ids"], 
        "attention_mask": encoding["attention_mask"], 
        "label": df["label"].values 
    })

In [9]:
train_dataset = to_dataset(df_train)
val_dataset = to_dataset(df_val) 

# set to torch format 
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load both BERT and RoBERTa models
They were trained on data formatted the same way: 
- Text cleaning
- Oversample the minority class for each keyword
- Add contextual information including the keyword and the country code to the text 

In [ ]:
import os 
# must pass abs path here 
bert_model = RobertaForSequenceClassification.from_pretrained(os.path.abspath("/vol/bitbucket/bm1325/NLP-PCL-Classification/models/model_only_oversample"), local_files_only = True)
trainer = Trainer(model = baseline_model)